In [1]:
# Importação de bibliotecas
import pandas as pd
import numpy as np

In [2]:
arquivo = "PRO-009-25_REV 1 - PROPOSTA COMERCIAL.xlsx" # nome do arquivo

aba = "CPU 1 " # aba = "CPU 1" # nome da aba

In [3]:
# Consumindo os dados
meta = pd.read_excel("PRO-009-25_REV 1 - PROPOSTA COMERCIAL.xlsx", sheet_name="CPU  1 ", nrows=2, header=None)

df = pd.read_excel("PRO-009-25_REV 1 - PROPOSTA COMERCIAL.xlsx", sheet_name="CPU  1 ", skiprows=5)

c:\Users\anderson.marley\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\anderson.marley\AppData\Local\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [4]:
# Buscando as colunas
df.columns = df.columns.map(str)

# Visualizando
df.columns

Index(['Unnamed: 0', 'Cód.', 'Discriminação', 'Unid', 'P.Unit', 'Quant',
       'F.utiliz', 'Custo', 'Quant.1', 'F.utiliz.1',
       ...
       'F.utiliz.36', 'Custo.36', 'Quant.37', 'F.utiliz.37', 'Custo.37',
       'Unnamed: 119', 'Quant.38', 'F.utiliz.38', 'Custo.38', 'Unnamed: 123'],
      dtype='object', length=124)

In [5]:
# Selecionando as colunas
lista = ['Unnamed: 0', 'Cód.', 'Discriminação', 'Unid', 'Unnamed: 123']

# Percorrendo as colunas e convertendo os tipos
for i in df.columns:
    
    # Checando se a coluna está na lista e Se estiver, converte para string, caso contrário, converte para numérico
    if i in lista:
        # Convertendo para string
        df[i] = df[i].astype(str)
        
    else:
        # Convertendo para numérico e arredondando
        df[i] = pd.to_numeric(df[i], errors='coerce').round(2)

In [6]:
#col_base = ['Cód.', 'Discriminação', 'Unid', 'P.Unit']

In [7]:
# Removando os NaN das colunas ...
df = df.dropna(subset=['Unid', 'P.Unit'])

In [8]:
# Alterando os dados NaN
df = df.fillna(0).round(2)

In [9]:
# Selecionando colunas bases
col_base = ['Cód.', 'Discriminação', 'Unid', 'P.Unit']

# Selecionando a partir da 5 coluna
col_inicial = 5

# Calcula o total de colunas do DF
col_total = len(df.columns)

# Elemina as colunas inicias e divide por 3 assumindo que cada grupo tem 3 colunas.
num_grupos = (col_total - col_inicial) // 3

In [10]:
# Criando tabela para composições
tabelas_composicoes = []

In [11]:
for i in range(num_grupos):

    # Calcula o indice atual, apos 'col_inicial'
    idx = col_inicial + i * 3
    
    # Pega a coluna da composição
    identificador = str(meta.iloc[0, idx]).strip()
    # Pega o nome da composição
    nome = str(meta.iloc[1, idx]).strip()
    
    # Pega os nomes das colunas
    col_quant = df.columns[idx]
    col_futiliz = df.columns[idx + 1]
    col_custo = df.columns[idx + 2]
    
    # Monta o bloco
    bloco = df[col_base + [col_quant, col_futiliz, col_custo]].copy()
    bloco.columns = col_base + ['Quant', 'Futiliz', 'Custo']
    bloco['Composicao'] = nome
    bloco['Identificador'] = identificador
    
    # Filtra apenas linhas relevantes (com valor em Quant ou Custo)
    bloco = bloco.dropna(subset=['Quant', 'Custo'], how='all')
    
    # Adiciona à lista final
    tabelas_composicoes.append(bloco)

In [12]:
# Concatenando todas as tabelas de composições
df_final = pd.concat(tabelas_composicoes, ignore_index=True)

In [13]:
# Organiza colunas
df_final = df_final[
    ['Cód.', 'Discriminação', 'Unid', 'P.Unit', 'Quant', 'Futiliz', 'Custo', 'Composicao']
]

In [14]:
# Exibe cada composição separadamente no Jupyter
composicoes_unicas = df_final['Composicao'].dropna().unique()

In [15]:
for comp in composicoes_unicas:
    print(f"\n🔹 Tabela: {comp}\n")

    # Filtra a composição atual
    tabela = df_final[df_final['Composicao'] == comp].copy()

    # Garante que a coluna 'Cód.' seja tratada corretamente
    cod_col = tabela['Cód.'].astype(str).str.strip().replace({'nan': np.nan, '': np.nan})

    # Encontra o primeiro índice onde 'Cód.' é NaN
    idx_nan = cod_col[cod_col.isna()].index

    # Se houver linha com NaN em 'Cód.', remove tudo a partir dela
    if not idx_nan.empty:
        tabela = tabela.loc[:idx_nan[0] - 1]

    # Exibe a tabela limpa
    display(tabela.reset_index(drop=True))



🔹 Tabela: Mobilização



,Cód.,Discriminação,Unid,P.Unit,Quant,Futiliz,Custo,Composicao
0,P01,Técnico SSMA,hh,30.03,160.0,1.00,4804.36,Mobilização
1,P02,Eletricista de força e controle,hh,35.64,160.0,1.00,5702.30,Mobilização
2,P03,Ajudante de elétrica,hh,21.21,160.0,2.00,6787.80,Mobilização
3,P04,Engenheiro eletricista,hh,122.13,0.0,0.00,0.00,Mobilização
4,P05,Engenheiro mecânico - Especialista em estrutur...,hh,122.13,0.0,0.00,0.00,Mobilização
5,P06,Projetista (Cadista),hh,28.53,0.0,0.00,0.00,Mobilização
6,P07,Alpinista industrial (Montador de andaime),hh,34.67,160.0,1.00,5546.75,Mobilização
7,P08,Encarregado de elétrica ou andaimes,hh,42.54,160.0,1.00,6806.18,Mobilização
8,P09,Montador de estruturas metálicas,hh,30.15,160.0,1.00,4823.26,Mobilização
9,P10,Topógrafo,hh,31.03,0.0,0.00,0.00,Mobilização



🔹 Tabela: Desmobilização



,Cód.,Discriminação,Unid,P.Unit,Quant,Futiliz,Custo,Composicao
0,P01,Técnico SSMA,hh,30.03,80.0,1.00,2402.18,Desmobilização
1,P02,Eletricista de força e controle,hh,35.64,80.0,1.00,2851.15,Desmobilização
2,P03,Ajudante de elétrica,hh,21.21,80.0,2.00,3393.90,Desmobilização
3,P04,Engenheiro eletricista,hh,122.13,0.0,0.00,0.00,Desmobilização
4,P05,Engenheiro mecânico - Especialista em estrutur...,hh,122.13,0.0,0.00,0.00,Desmobilização
5,P06,Projetista (Cadista),hh,28.53,0.0,0.00,0.00,Desmobilização
6,P07,Alpinista industrial (Montador de andaime),hh,34.67,80.0,1.00,2773.37,Desmobilização
7,P08,Encarregado de elétrica ou andaimes,hh,42.54,80.0,1.00,3403.09,Desmobilização
8,P09,Montador de estruturas metálicas,hh,30.15,80.0,1.00,2411.63,Desmobilização
9,P10,Topógrafo,hh,31.03,0.0,0.00,0.00,Desmobilização



🔹 Tabela: Canteiro de Obras



...

,Cód.,Discriminação,Unid,P.Unit,Quant,Futiliz,Custo,Composicao
0,P01,Técnico SSMA,hh,30.03,40.0,1.00,1201.09,Montagem da monovia existente
1,P02,Eletricista de força e controle,hh,35.64,0.0,0.00,0.00,Montagem da monovia existente
2,P03,Ajudante de elétrica,hh,21.21,0.0,0.00,0.00,Montagem da monovia existente
3,P04,Engenheiro eletricista,hh,122.13,0.0,0.00,0.00,Montagem da monovia existente
4,P05,Engenheiro mecânico - Especialista em estrutur...,hh,122.13,0.0,0.00,0.00,Montagem da monovia existente
5,P06,Projetista (Cadista),hh,28.53,0.0,0.00,0.00,Montagem da monovia existente
6,P07,Alpinista industrial (Montador de andaime),hh,34.67,40.0,1.00,1386.69,Montagem da monovia existente
7,P08,Encarregado de elétrica ou andaimes,hh,42.54,40.0,1.00,1701.55,Montagem da monovia existente
8,P09,Montador de estruturas metálicas,hh,30.15,40.0,1.00,1205.82,Montagem da monovia existente
9,P10,Topógrafo,hh,31.03,0.0,0.00,0.00,Montagem da monovia existente



🔹 Tabela: Montagem das estruturas transversais de apoio



,Cód.,Discriminação,Unid,P.Unit,Quant,Futiliz,Custo,Composicao
0,P01,Técnico SSMA,hh,30.03,24.0,1.00,720.65,Montagem das estruturas transversais de apoio
1,P02,Eletricista de força e controle,hh,35.64,0.0,0.00,0.00,Montagem das estruturas transversais de apoio
2,P03,Ajudante de elétrica,hh,21.21,0.0,0.00,0.00,Montagem das estruturas transversais de apoio
3,P04,Engenheiro eletricista,hh,122.13,0.0,0.00,0.00,Montagem das estruturas transversais de apoio
4,P05,Engenheiro mecânico - Especialista em estrutur...,hh,122.13,0.0,0.00,0.00,Montagem das estruturas transversais de apoio
5,P06,Projetista (Cadista),hh,28.53,0.0,0.00,0.00,Montagem das estruturas transversais de apoio
6,P07,Alpinista industrial (Montador de andaime),hh,34.67,24.0,1.00,832.01,Montagem das estruturas transversais de apoio
7,P08,Encarregado de elétrica ou andaimes,hh,42.54,24.0,1.00,1020.93,Montagem das estruturas transversais de apoio
8,P09,Montador de estruturas metálicas,hh,30.15,24.0,1.00,723.49,Montagem das estruturas transversais de apoio
9,P10,Topógrafo,hh,31.03,0.0,0.00,0.00,Montagem das estruturas transversais de apoio



🔹 Tabela: nan



,Cód.,Discriminação,Unid,P.Unit,Quant,Futiliz,Custo,Composicao
0,P01,Técnico SSMA,hh,30.03,0.0,0.0,0.0,nan
1,P02,Eletricista de força e controle,hh,35.64,0.0,0.0,0.0,nan
2,P03,Ajudante de elétrica,hh,21.21,0.0,0.0,0.0,nan
3,P04,Engenheiro eletricista,hh,122.13,0.0,0.0,0.0,nan
4,P05,Engenheiro mecânico - Especialista em estrutur...,hh,122.13,0.0,0.0,0.0,nan
5,P06,Projetista (Cadista),hh,28.53,0.0,0.0,0.0,nan
6,P07,Alpinista industrial (Montador de andaime),hh,34.67,0.0,0.0,0.0,nan
7,P08,Encarregado de elétrica ou andaimes,hh,42.54,0.0,0.0,0.0,nan
8,P09,Montador de estruturas metálicas,hh,30.15,0.0,0.0,0.0,nan
9,P10,Topógrafo,hh,31.03,0.0,0.0,0.0,nan
